# Stop location, accompanying stops: multinomial logit over 1,588 zones, sequential anchors, location mixture (article table est_stop_ACCOMPANY_stop)

Re-estimate the location choice of accompanying stops on tours and check it against the article.

This notebook goes from the raw data to the estimates of this model:

1. Raw data
2. Data preparation
3. Estimation data
4. Descriptive statistics
5. Correlation of the explanatory variables (heatmaps, VIF)
6. Model specification (the folder's `model/estimate_stop_accompanying.py`, unchanged)
7. Log-likelihood at the article coefficients
8. Biogeme estimation
9. Results: article format, fit, comparison with the article, correlation of the estimates
10. Hold-out sample

**Kernel:** select *Python (biogeme)* (`~/.venvs/biogeme`) in VS Code, then *Run All*. The switches in the next cell decide what is recomputed.

In [ ]:
RUN_DATA_PREP  = False   # True: rebuild the estimation tables (section 2)
RUN_CHECK      = False    # log-likelihood at the article coefficients  (this model: longer than 30 min)
RUN_ESTIMATION = False    # Biogeme estimation, written to output/  (this model: beyond the memory of this machine, see RUN.json notes; False reads output/ if a result is there)
RUN_HOLDOUT    = False    # log-likelihood of the hold-out half at the full-precision estimates

import os, sys, json, glob, inspect, importlib, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from biogeme.results_processing import EstimationResults

FOLDER = os.path.abspath("")                       # this model folder (VS Code starts the notebook here)
sys.path.insert(0, os.path.join(FOLDER, "..", "..", "tools")); import nbtools as nt
sys.path.insert(0, os.path.join(FOLDER, "model"))
RUN = json.load(open(os.path.join(FOLDER, "RUN.json")))
OUT = os.path.join(FOLDER, "output"); os.makedirs(OUT, exist_ok=True)
NAME = "stop_accompanying"
pd.set_option("display.max_rows", 200, "display.max_columns", 60, "display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("model folder:", FOLDER)

## 1. Raw data

Where each source comes from and where it goes (paths from `manual_estimation/`). *verify*: no download link is recorded in the project. The full download steps are in `INSTRUCTIONS.pdf`.

In [ ]:
raw = os.path.join(FOLDER, "input", "data", "raw")
if os.path.isdir(raw):
    display(nt.files_table(raw))
    for f in sorted(glob.glob(os.path.join(raw, "*")))[:6]:
        head, n = nt.preview(f)
        if head is not None:
            print(f"\n{os.path.basename(f)}" + (f"  ({n:,} rows)" if n else ""))
            display(head)
shared = os.path.join(FOLDER, "..", "00_shared_data")
print("shared data folders:", sorted(os.listdir(shared)))

## 2. Data preparation

Shared and upstream stages (outputs provided in `../00_shared_data` and `input/data/raw`; run them only for a full rebuild from the raw data; `cd` paths from this model folder):


Package steps, run below when `RUN_DATA_PREP = True`: they rebuild `input/data/estimation` and `input/data/holdout`; `check_against_provided.py` compares the rebuilt tables with the provided ones.

**Step 1.** optional (data already provided in input/data/estimation and input/data/holdout): estimation tables of the stops from the raw TAZ record files (about 0.5 min)

In [ ]:
if RUN_DATA_PREP:
    nt.run('PYTHONDONTWRITEBYTECODE=1 python3 build_estimation_tables.py', cwd=os.path.join(FOLDER, 'input/code'))
else:
    print("skipped (RUN_DATA_PREP = False): the provided tables are used")

**Step 2.** optional (data already provided): wide Biogeme data: detour and mode-choice logsum of every anchor configuration to every zone (about 1.5 min)

In [ ]:
if RUN_DATA_PREP:
    nt.run('PYTHONDONTWRITEBYTECODE=1 python3 build_biogeme_data.py', cwd=os.path.join(FOLDER, 'input/code'))
else:
    print("skipped (RUN_DATA_PREP = False): the provided tables are used")

## 3. Estimation data

The tables exactly as used in the estimation, and the data dictionary.

In [ ]:
import estimate_stop_accompanying as M
est = os.path.join(FOLDER, "input", "data", "estimation")
display(nt.files_table(est))
dd = pd.read_csv(os.path.join(est, "data_dictionary.csv")) if os.path.exists(os.path.join(est, "data_dictionary.csv")) else None
dd

In [ ]:
# Biogeme table: one row per record; imp_<c>_<zone>, cgt_<c>_<zone> = zone variables of configuration c for all zones,
# cand_* / b_* = the candidate zones of the observed location and their weights. Only the first two parts are read here.
meta = json.load(open(os.path.join(est, "biogeme_meta.json")))
print({k: v for k, v in meta.items() if k != "parts"})
parts = sorted(glob.glob(os.path.join(est, "biogeme_part*.parquet")))
df = pd.concat([pd.read_parquet(p) for p in parts[:2]], ignore_index=True)
print(f"{len(parts)} parts; read {len(df):,} of {meta['N']:,} records, {df.shape[1]:,} columns")
rec_f = os.path.join(est, "records.parquet")
records = pd.read_parquet(rec_f) if os.path.exists(rec_f) else None
display(records.head() if records is not None else df.iloc[:5, :30])
ZONES = pd.read_csv(os.path.join(FOLDER, "..", "00_shared_data", "zones", "zones_model_attributes.csv"))

## 4. Descriptive statistics

In [ ]:
person = nt.explanatory(df[[c for c in df.columns if c.startswith("f_")]])
display(person.describe().T.round(3))
if "purpose_id" in df and records is not None and "purpose" in records:
    display(records["purpose"].value_counts().rename("records").to_frame())
if "n_cand" in df:
    ax = df["n_cand"].value_counts().sort_index().plot.bar(figsize=(5, 3), color="#2b6cb0", rot=0)
    ax.set_xlabel("candidate zones of the observed location"); ax.set_ylabel("records"); plt.show()
if records is not None:
    display(nt.explanatory(records).describe().T.round(3))

## 5. Model specification

The folder's script `model/estimate_stop_accompanying.py`, imported as it is. Its utility functions:

In [ ]:
src = inspect.getsource(M)
print(src[:src.find("def read(")])

## 6. Correlation of the explanatory variables

|r| ≥ 0.7 and VIF ≥ 10 are listed. Interaction terms are correlated with their main effects by construction.

In [ ]:
long = nt.explanatory(nt.location_long(df, ZONES))
print(f"zone variables of {min(300, len(df))} records x {len(ZONES):,} zones = {len(long):,} rows")
Cz = nt.corr_heatmap(long, "Destination variables (impedance, generalised time, zone attributes)")
display(nt.high_pairs(Cz)); display(nt.vif(long))
Cp = nt.corr_heatmap(person, "Person and tour variables of the records"); display(nt.high_pairs(Cp))

## 7. Log-likelihood at the article coefficients

No estimation: the script's `evaluate("estimation", "article")`. Target: article -12417.5,
Biogeme -12417.4835 (N = 3515, K = 26).

In [ ]:
def evaluate(half, at):
    here = os.getcwd()
    try:
        return M.evaluate(half, at) if "at" in inspect.signature(M.evaluate).parameters else M.evaluate(half)
    finally:
        os.chdir(here)

if RUN_CHECK:
    t0 = time.time(); ll_article = evaluate("estimation", "article"); print(f"{time.time()-t0:.0f} s")
else:
    print("skipped (RUN_CHECK = False)")

## 8. Biogeme estimation

The script's `estimate()`: Biogeme with its defaults (`output/biogeme.toml`), robust standard errors. Report in
`output/stop_accompanying.html` and `.yaml`; table `output/estimated_parameters.csv`.

In [ ]:
if RUN_ESTIMATION:
    here = os.getcwd(); t0 = time.time()
    try:
        M.estimate(OUT)
    finally:
        os.chdir(here)
    print(f"estimation time {time.time()-t0:.0f} s")
y = [f for f in sorted(glob.glob(os.path.join(OUT, "*.yaml")), key=os.path.getmtime) if "eval" not in os.path.basename(f)]
results = EstimationResults.from_yaml_file(y[-1]) if y else None
print(f"results read from {os.path.basename(y[-1])}" if y else "no result in output/: sections 9-10 are skipped")

## 9. Results

### Estimates in the article format
Coefficient (stars: *** p < 0.001, ** p < 0.01, * p < 0.05), robust s.e., p.

In [ ]:
if results is not None:
    est_t = nt.estimates_table(results)
    display(nt.article_format(est_t))

### Goodness of fit

In [ ]:
if results is not None:
    display(nt.fit_statistics(results))

### Comparison with the article (`model/reference_estimates.csv`)

In [ ]:
if results is not None:
    REF = os.path.join(FOLDER, "model", "reference_estimates.csv")
    cmp = nt.compare(est_t, REF)
    cmp.to_csv(os.path.join(OUT, "comparison_with_article_notebook.csv"), index=False)
    print(f"{int(cmp.matches_article.sum())} of {len(cmp)} parameters match the article (coefficient and robust s.e. within the 4th-decimal rounding); "
          f"largest |difference|: coefficient {cmp.diff_coef.abs().max():.5f}, s.e. {cmp.diff_se.abs().max():.5f}")
    display(cmp.round({"coefficient": 4, "robust_se": 4, "p": 4, "diff_coef": 5, "diff_se": 5}))
    stats = nt.article_statistics(REF)
    if len(stats):
        stats["Biogeme"] = stats["statistic"].map({"LL(beta)": results.final_loglikelihood, "K": results.number_of_free_parameters, "N": 3515})
        display(stats)
    nt.plot_against_article(cmp, RUN.get("model", NAME))

### Correlation of the estimates
Robust correlation matrix of the estimated parameters; pairs with |r| ≥ 0.7 listed.

In [ ]:
if results is not None:
    display(nt.parameter_correlation(results, "Robust correlation of the estimates"))

## 10. Hold-out sample

Log-likelihood of the hold-out half at the full-precision estimates (`evaluate("holdout", "full")`).

In [ ]:
if RUN_HOLDOUT:
    ll_hold = evaluate("holdout", "full")
else:
    print("skipped (RUN_HOLDOUT = False)")